# Healthcare Data Analysis: Breast Cancer Diagnosis Prediction
### IBM SkillsBuild Internship — Data Analytics Project

**Dataset:** Breast Cancer Wisconsin (Diagnostic) Data Set (via `sklearn.datasets`, originally UCI ML Repository)

**Objective:** Explore cell-nuclei measurements taken from breast mass biopsies, understand how they relate to
diagnosis, and build a classification model that predicts whether a tumor is **Malignant** or **Benign**.

**Workflow:**
1. Data Loading & Understanding
2. Data Cleaning & Preprocessing
3. Exploratory Data Analysis (EDA)
4. Model Building (Logistic Regression & Random Forest)
5. Model Evaluation & Comparison
6. Conclusions & Insights


## 1. Import Libraries

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, roc_curve, roc_auc_score,
                              classification_report)

sns.set_style("whitegrid")
%matplotlib inline


## 2. Load the Dataset

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df["diagnosis"] = df["target"].map({0: "Malignant", 1: "Benign"})
df.to_csv("breast_cancer_data.csv", index=False)
df.head()


## 3. Data Understanding & Cleaning

In [ ]:
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes.value_counts())
print("\nTotal missing values:", df.isnull().sum().sum())
print("\nClass balance:\n", df["diagnosis"].value_counts())


In [ ]:
df.describe().T


**Observation:** The dataset contains 569 records and 30 numeric features (no missing values), describing
cell-nuclei characteristics such as radius, texture, perimeter, area, smoothness, and concavity, computed for
each biopsy image. The target classes are reasonably balanced (357 Benign vs 212 Malignant).

## 4. Exploratory Data Analysis (EDA)

### 4.1 Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x="diagnosis", hue="diagnosis", data=df, palette=["#d95f5f", "#5fa8d9"], legend=False, ax=ax)
ax.set_title("Diagnosis Class Distribution")
ax.set_xlabel("")
plt.tight_layout()
plt.show()


### 4.2 Correlation Heatmap

In [ ]:
corr = df.drop(columns=["target", "diagnosis"]).corr()
top_feats = corr["mean radius"].abs().sort_values(ascending=False).index[:10]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[top_feats].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Correlation Heatmap (Top Features Related to Mean Radius)")
plt.tight_layout()
plt.show()


### 4.3 Feature Distributions by Diagnosis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
feats = ["mean radius", "mean texture", "mean concavity", "mean smoothness"]
for ax, f in zip(axes.flat, feats):
    sns.boxplot(x="diagnosis", y=f, hue="diagnosis", data=df, palette=["#d95f5f", "#5fa8d9"], legend=False, ax=ax)
    ax.set_title(f)
    ax.set_xlabel("")
fig.suptitle("Key Feature Distributions by Diagnosis")
plt.tight_layout()
plt.show()


### 4.4 Feature Relationship

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(x="mean radius", y="mean concave points", hue="diagnosis",
                 data=df, palette=["#d95f5f", "#5fa8d9"], alpha=0.7, ax=ax)
ax.set_title("Mean Radius vs Mean Concave Points")
plt.tight_layout()
plt.show()


**Insight:** Malignant tumors tend to have consistently higher mean radius, texture, concavity, and
concave-points values than benign tumors, with `mean concave points` and `mean radius` showing a clear
separating trend — both are strong candidate predictors.

## 5. Data Preprocessing

In [ ]:
X = df[data.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 6. Model Building

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

results = {}
roc_data = {}

for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    probs = model.predict_proba(X_test_s)[:, 1]

    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1_score": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, probs),
    }
    roc_data[name] = roc_curve(y_test, probs)[:2]

pd.DataFrame(results).T


## 7. Model Evaluation

### 7.1 Confusion Matrix (Random Forest)

In [ ]:
rf_model = models["Random Forest"]
rf_preds = rf_model.predict(X_test_s)
cm = confusion_matrix(y_test, rf_preds)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Malignant", "Benign"], yticklabels=["Malignant", "Benign"], ax=ax)
ax.set_title("Confusion Matrix - Random Forest")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

print(classification_report(y_test, rf_preds, target_names=["Malignant", "Benign"]))


### 7.2 Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=data.feature_names)
top10 = importances.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=top10.values, y=top10.index, hue=top10.index, palette="viridis", legend=False, ax=ax)
ax.set_title("Top 10 Feature Importances - Random Forest")
plt.tight_layout()
plt.show()


### 7.3 ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, (fpr, tpr) in roc_data.items():
    ax.plot(fpr, tpr, label=f"{name} (AUC={results[name]['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison")
ax.legend()
plt.tight_layout()
plt.show()


## 8. Save Metrics

In [ ]:
with open("metrics.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))


## 9. Conclusions

- Both models achieved strong performance, with **Logistic Regression** slightly outperforming
  **Random Forest** on this dataset (higher accuracy, F1-score, and ROC-AUC).
- The most influential features for classification were related to **cell concavity, area, and
  perimeter** — larger and more irregularly shaped nuclei were strongly associated with malignant tumors.
- The high recall achieved on the malignant class is particularly important in a healthcare context, since
  minimizing false negatives (missed malignant cases) is critical.
- **Future work:** hyperparameter tuning (GridSearchCV), cross-validation, trying additional algorithms
  (SVM, XGBoost), and testing on external/unseen clinical data to validate generalizability.
